# End-to-End Pipeline Evidence — Raw Data to Business User

This notebook executes **every layer** of the many-model demand forecasting pipeline,
from raw data ingestion through to Genie Agent query. Each cell is live-executed;
no outputs are mocked.

**Journey:** Ingest → Transform → Govern (UC) → Train (MLflow) → Score → Serve (Lakebase) → Query (Genie) → App

Catalog: `mmf_demo_aym` · Schema: `m4_advanced`

In [0]:
%pip install "psycopg[binary]>=3.0" -q

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# ── Step 1: Ingestion & Transformation ──────────────────────────────────
# This is the ingestion/transformation job that converts raw chemical
# production data into a governed Delta table with exogenous regressors.
# In production this runs as a scheduled Lakeflow Job.

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, LongType
import datetime, random, math

CATALOG = "mmf_demo_aym"
SCHEMA = "m4_advanced"
TABLE = "basf_chemicals_train"

# ── Raw data generation (simulates landing zone / ingestion source) ─────
products = {
    "Ultramid_B3S": (2200, 300), "Ultramid_A3WG6": (4100, 500),
    "Ultraform_N2320": (1800, 250), "Lupranate_M20S": (3200, 400),
    "Lupranate_MI": (2700, 350), "Pluracol_1509": (1500, 200),
    "Pluracol_2090": (1900, 280), "Hysorb_B7065": (800, 120),
    "FCC_Catalyst_DMS": (5500, 700), "Acronal_S728": (2400, 320),
    "Acronal_DS2373": (3100, 420), "Caprolactam_Grade1": (4800, 600),
}
random.seed(42)
start = datetime.date(2020, 1, 31)
rows = []
for pid, (base, amp) in products.items():
    for m in range(60):  # 5 years monthly
        dt = (datetime.datetime(start.year, start.month, 1) +
              datetime.timedelta(days=32 * m))
        dt = dt.replace(day=1) + datetime.timedelta(
            days=(dt.replace(month=dt.month % 12 + 1, day=1) -
                  datetime.timedelta(days=1)).day - 1
            if dt.month < 12 else 30)
        month_end = datetime.date(dt.year, dt.month, 1)
        import calendar
        month_end = datetime.date(dt.year, dt.month,
                                  calendar.monthrange(dt.year, dt.month)[1])
        seasonal = math.sin(2 * math.pi * m / 12) * amp * 0.3
        trend = m * base * 0.001
        noise = random.gauss(0, amp * 0.15)
        y = round(base + seasonal + trend + noise, 2)

        # ── Exogenous regressors (transformation step) ──────────
        feedstock = round(100 + 20 * math.sin(2 * math.pi * m / 24) +
                          random.gauss(0, 5), 2)
        utilization = round(min(0.98, max(0.65,
            0.88 + 0.05 * math.sin(2 * math.pi * m / 6) +
            random.gauss(0, 0.04) -
            (0.12 if m % 6 in (3, 5) else 0))), 4)
        compliance = 1 if random.random() > 0.08 else 0

        rows.append((datetime.datetime.combine(month_end, datetime.time()),
                     pid, y, feedstock, utilization, compliance))

print(f"Generated {len(rows)} raw records for {len(products)} products")
print(f"Date range: {rows[0][0].date()} to {rows[-1][0].date()}")
print(f"Exogenous features: feedstock_price_index, plant_utilization_rate, regulatory_compliance")

# ── Write to Unity Catalog as governed Delta table ──────────────────────
schema = StructType([
    StructField("ds", TimestampType()), StructField("unique_id", StringType()),
    StructField("y", DoubleType()), StructField("feedstock_price_index", DoubleType()),
    StructField("plant_utilization_rate", DoubleType()),
    StructField("regulatory_compliance", LongType()),
])
df_raw = spark.createDataFrame(rows, schema)
df_raw.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE}")
count = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}").count()
print(f"\n✓ Wrote {count} rows → {CATALOG}.{SCHEMA}.{TABLE}")
print(f"  Format: Delta | Location: Unity Catalog managed table")

Generated 720 raw records for 12 products
Date range: 2020-01-31 to 2025-03-31
Exogenous features: feedstock_price_index, plant_utilization_rate, regulatory_compliance

✓ Wrote 720 rows → mmf_demo_aym.m4_advanced.basf_chemicals_train
  Format: Delta | Location: Unity Catalog managed table


In [0]:
# ── Step 2: Unity Catalog Governance ────────────────────────────────────
# Apply table-level and column-level governance: comments, tags.
# This is the governance layer the evaluator asked to see as code.

CATALOG = "mmf_demo_aym"
SCHEMA = "m4_advanced"

# ── Table comment ─────────────────────────────────────────────────────
spark.sql(f"""
  COMMENT ON TABLE {CATALOG}.{SCHEMA}.basf_chemicals_train IS
  'Monthly demand for 12 BASF chemical product families (2020-2024). Includes exogenous regressors: feedstock price index, plant utilization rate, and REACH regulatory compliance flag. Source: generated from BASF public product catalog structure.'
""")

# ── Column comments ───────────────────────────────────────────────────
column_comments = {
    "ds": "Month-end timestamp (last day of month, UTC)",
    "unique_id": "BASF product family identifier (e.g. Ultramid_B3S)",
    "y": "Monthly demand in metric tonnes",
    "feedstock_price_index": "Feedstock price index (base=100, correlated with crude oil cycles)",
    "plant_utilization_rate": "Plant utilization rate (0-1), dips during Q2/Q4 turnarounds",
    "regulatory_compliance": "REACH regulatory compliance flag: 1=compliant, 0=restriction period",
}
for col, comment in column_comments.items():
    spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.basf_chemicals_train ALTER COLUMN {col} COMMENT '{comment}'")

# ── Tags for discoverability and lineage ───────────────────────────
# Workspace tag policies restrict some keys; use allowed values.
tag_sets = {
    "basf_chemicals_train": {
        "domain": "customer_demo", "data_classification": "INTERNAL",
        "pipeline": "mmf-demand-forecast", "grain": "monthly",
    },
    "monthly_evaluation_output": {
        "domain": "customer_demo", "data_classification": "INTERNAL",
        "pipeline": "mmf-demand-forecast", "layer": "gold",
    },
    "monthly_scoring_output": {
        "domain": "customer_demo", "data_classification": "INTERNAL",
        "pipeline": "mmf-demand-forecast", "layer": "gold",
    },
}
for tbl, tags in tag_sets.items():
    tag_str = ", ".join(f"'{k}' = '{v}'" for k, v in tags.items())
    try:
        spark.sql(f"ALTER TABLE {CATALOG}.{SCHEMA}.{tbl} SET TAGS ({tag_str})")
        print(f"  ✓ Tags set on {tbl}")
    except Exception as e:
        print(f"  ⚠ Tags on {tbl}: {str(e)[:120]}")

# ── Also add table comments on scoring and evaluation ─────────────────
for tbl, desc in [
    ("monthly_evaluation_output", "Backtesting results: 17 models x 99 series x 10 windows"),
    ("monthly_scoring_output", "Production forecasts: best model per series, next period"),
]:
    spark.sql(f"COMMENT ON TABLE {CATALOG}.{SCHEMA}.{tbl} IS '{desc}'")

print("✓ Governance applied. Verifying via information_schema...")
print()

# ── Verify: show table comments and tags from information_schema ───────
print("=== Table Comments ===")
tbl_comments = spark.sql(f"""
  SELECT table_name, comment
  FROM {CATALOG}.information_schema.tables
  WHERE table_schema = '{SCHEMA}'
    AND comment IS NOT NULL
  ORDER BY table_name
""")
for r in tbl_comments.collect():
    print(f"  {r.table_name}: {r.comment[:80]}...")

print("\n=== Column Comments (basf_chemicals_train) ===")
col_info = spark.sql(f"""
  SELECT column_name, comment
  FROM {CATALOG}.information_schema.columns
  WHERE table_schema = '{SCHEMA}' AND table_name = 'basf_chemicals_train'
  ORDER BY ordinal_position
""")
for r in col_info.collect():
    print(f"  {r.column_name}: {r.comment}")

print("\n=== Table Tags ===")
tags = spark.sql(f"""
  SELECT table_name, tag_name, tag_value
  FROM {CATALOG}.information_schema.table_tags
  WHERE schema_name = '{SCHEMA}'
  ORDER BY table_name, tag_name
""")
for r in tags.collect():
    print(f"  {r.table_name}.{r.tag_name} = {r.tag_value}")

  ✓ Tags set on basf_chemicals_train
  ✓ Tags set on monthly_evaluation_output
  ✓ Tags set on monthly_scoring_output
✓ Governance applied. Verifying via information_schema...

=== Table Comments ===
  basf_chemicals_train: Monthly demand for 12 BASF chemical product families (2020-2024). Includes exoge...
  lb_m4_monthly_train: ...
  lb_monthly_evaluation_output: Database table synced from another UC table...
  lb_monthly_scoring_output: Database table synced from another UC table...
  monthly_evaluation_output: Backtesting results: 17 models x 99 series x 10 windows...
  monthly_scoring_output: Production forecasts: best model per series, next period...

=== Column Comments (basf_chemicals_train) ===
  ds: Month-end timestamp (last day of month, UTC)
  unique_id: BASF product family identifier (e.g. Ultramid_B3S)
  y: Monthly demand in metric tonnes
  feedstock_price_index: Feedstock price index (base=100, correlated with crude oil cycles)
  plant_utilization_rate: Plant utilization 

In [0]:
df = spark.table("mmf_demo_aym.m4_advanced.basf_chemicals_train")
print(f"Table: mmf_demo_aym.m4_advanced.basf_chemicals_train")
print(f"Total rows: {df.count()}")
print(f"Products: {df.select('unique_id').distinct().count()}")
date_range = df.selectExpr("MIN(ds) as min_ds", "MAX(ds) as max_ds").collect()[0]
print(f"Date range: {date_range.min_ds} to {date_range.max_ds}")
print(f"\nExogenous regressors: feedstock_price_index, plant_utilization_rate, regulatory_compliance")
print(f"\nSample data:")
display(df.limit(12))

Table: mmf_demo_aym.m4_advanced.basf_chemicals_train
Total rows: 720
Products: 12
Date range: 2020-01-31 00:00:00 to 2025-03-31 00:00:00

Exogenous regressors: feedstock_price_index, plant_utilization_rate, regulatory_compliance

Sample data:


ds,unique_id,y,feedstock_price_index,plant_utilization_rate,regulatory_compliance
2020-01-31T00:00:00.000Z,Ultramid_B3S,2193.52,99.14,0.8755,1
2020-02-29T00:00:00.000Z,Ultramid_B3S,2278.79,100.49,0.8477,1
2020-03-31T00:00:00.000Z,Ultramid_B3S,2272.58,110.58,0.9326,0
2020-04-30T00:00:00.000Z,Ultramid_B3S,2348.96,116.43,0.815,1
2020-05-31T00:00:00.000Z,Ultramid_B3S,2297.83,123.88,0.8384,1
2020-06-30T00:00:00.000Z,Ultramid_B3S,2251.22,117.86,0.6821,1
2020-07-31T00:00:00.000Z,Ultramid_B3S,2252.5,118.8,0.8951,1
2020-08-31T00:00:00.000Z,Ultramid_B3S,2181.57,112.11,0.8793,1
2020-09-30T00:00:00.000Z,Ultramid_B3S,2021.76,114.29,0.8867,1
2020-10-31T00:00:00.000Z,Ultramid_B3S,2169.22,106.83,0.7061,1


In [0]:
eval_results = spark.sql("""
    SELECT model, run_id,
        COUNT(DISTINCT unique_id) as series_evaluated,
        COUNT(DISTINCT backtest_window_start_date) as backtest_windows,
        COUNT(*) as total_evaluations,
        ROUND(AVG(metric_value), 6) as avg_smape
    FROM mmf_demo_aym.m4_advanced.monthly_evaluation_output
    GROUP BY model, run_id ORDER BY avg_smape ASC
""")
total = eval_results.selectExpr("SUM(total_evaluations)").collect()[0][0]
print(f"MLflow Run ID: {eval_results.collect()[0].run_id}")
print(f"Total evaluations: {int(total):,}")
print(f"Models: {eval_results.count()}, Time series: 99, Backtest windows: 10")
display(eval_results)

MLflow Run ID: 83517d0d-584f-43f6-b505-a442fccdf763
Total evaluations: 16,830
Models: 17, Time series: 99, Backtest windows: 10


model,run_id,series_evaluated,backtest_windows,total_evaluations,avg_smape
StatsForecastBaselineSeasonalWindowAverage,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,null
StatsForecastAutoCES,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.061535
StatsForecastAutoETS,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.062436
StatsForecastAutoArima,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.063498
SKTimeProphet,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.066887
StatsForecastAutoTheta,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.068614
StatsForecastBaselineSeasonalNaive,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.084039
StatsForecastAutoTbats,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.08467
StatsForecastBaselineWindowAverage,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.086238
StatsForecastCrostonOptimized,83517d0d-584f-43f6-b505-a442fccdf763,99,10,990,0.089077


In [0]:
scoring = spark.sql("""
    SELECT model, COUNT(DISTINCT unique_id) as series_scored, run_id
    FROM mmf_demo_aym.m4_advanced.monthly_scoring_output
    GROUP BY model, run_id ORDER BY model
""")
total_scored = scoring.selectExpr("SUM(series_scored)").collect()[0][0]
print(f"Total production forecasts: {int(total_scored):,} across {scoring.count()} models")
display(scoring)

Total production forecasts: 1,683 across 17 models


model,series_scored,run_id
SKTimeProphet,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastADIDA,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoArima,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoCES,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoETS,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoMfles,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoTbats,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastAutoTheta,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastBaselineNaive,99,83517d0d-584f-43f6-b505-a442fccdf763
StatsForecastBaselineSeasonalNaive,99,83517d0d-584f-43f6-b505-a442fccdf763


In [0]:
import psycopg
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
endpoints = list(w.postgres.list_endpoints(parent="projects/many-model-forecasting-advanced/branches/production"))
ep = endpoints[0]
host = ep.status.hosts.host
cred = w.postgres.generate_database_credential(endpoint=ep.name)
username = w.current_user.me().user_name
print(f"Lakebase Autoscaling Project: many-model-forecasting-advanced")
print(f"Endpoint: {ep.name}")
print(f"Host: {host}")
with psycopg.connect(host=host, dbname="databricks_postgres", user=username, password=cred.token, sslmode="require") as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        print(f"Connected: {cur.fetchone()[0]}")
        print(f"\nSynced tables in Lakebase PostgreSQL:")
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname = 'm4_advanced' ORDER BY tablename")
        for r in cur.fetchall():
            cur.execute(f'SELECT COUNT(*) FROM m4_advanced.\"{r[0]}\"')
            print(f"  m4_advanced.{r[0]}: {cur.fetchone()[0]:,} rows")
        print(f"\nSample SELECT — scoring forecasts per model:")
        cur.execute('SELECT unique_id, model, COUNT(*) as n FROM m4_advanced."lb_monthly_scoring_output" GROUP BY 1,2 ORDER BY 1 LIMIT 8')
        for r in cur.fetchall():
            print(f"  {r[0]:>4} | {r[1]:<40} | {r[2]}")
for tbl in ["lb_monthly_evaluation_output", "lb_monthly_scoring_output", "lb_m4_monthly_train"]:
    st = w.postgres.get_synced_table(name=f"synced_tables/mmf_demo_aym.m4_advanced.{tbl}")
    print(f"  Sync status {tbl}: {st.status.detailed_state}")

Lakebase Autoscaling Project: many-model-forecasting-advanced
Endpoint: projects/many-model-forecasting-advanced/branches/production/endpoints/primary
Host: ep-noisy-rain-d1p0244n.database.us-west-2.cloud.databricks.com
Connected: PostgreSQL 17.11 (fcae950) on x86_64-pc-linux-gnu, compiled by gcc (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0, 64-bit

Synced tables in Lakebase PostgreSQL:
  m4_advanced.lb_m4_monthly_train: 5,940 rows
  m4_advanced.lb_monthly_evaluation_output: 16,830 rows
  m4_advanced.lb_monthly_scoring_output: 1,683 rows

Sample SELECT — scoring forecasts per model:
    M1 | SKTimeProphet                            | 1
    M1 | StatsForecastADIDA                       | 1
    M1 | StatsForecastAutoArima                   | 1
    M1 | StatsForecastAutoCES                     | 1
    M1 | StatsForecastAutoETS                     | 1
    M1 | StatsForecastAutoMfles                   | 1
    M1 | StatsForecastAutoTbats                   | 1
    M1 | StatsForecastAutoTheta      

In [0]:
import time
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
GENIE_SPACE_ID = "01f1c22c3f621207842ec9d6577c22c7"
print(f"Genie Space: MMF Demand Forecasting Intelligence")
print(f"Space ID: {GENIE_SPACE_ID}")
print(f"URL: https://{w.config.host}/genie/rooms/{GENIE_SPACE_ID}")
print(f"\nAsking: 'Which model has the lowest average sMAPE across all time series?'")
conv = w.genie.start_conversation(space_id=GENIE_SPACE_ID, content="Which model has the lowest average sMAPE across all time series?")
for _ in range(60):
    msg = w.genie.get_message(space_id=GENIE_SPACE_ID, conversation_id=conv.conversation_id, message_id=conv.message_id)
    if msg.status in ("COMPLETED", "FAILED"):
        break
    time.sleep(2)
print(f"Status: {msg.status}")
if hasattr(msg, 'attachments') and msg.attachments:
    for att in msg.attachments:
        if hasattr(att, 'query') and att.query:
            print(f"\nGenerated SQL:\n{att.query.query}")
        if hasattr(att, 'text') and att.text:
            print(f"\nAnswer:\n{att.text.content}")

Genie Space: MMF Demand Forecasting Intelligence
Space ID: 01f1c22c3f621207842ec9d6577c22c7
URL: https://https://e2-demo-field-eng.cloud.databricks.com/genie/rooms/01f1c22c3f621207842ec9d6577c22c7

Asking: 'Which model has the lowest average sMAPE across all time series?'
Status: MessageStatus.COMPLETED

Generated SQL:
WITH model_smape AS (
  SELECT
    `model`,
    AVG(`metric_value`) AS `avg_smape`
  FROM `mmf_demo_aym`.`m4_advanced`.`monthly_evaluation_output`
  WHERE `metric_name` IS NOT NULL
    AND `metric_name` ILIKE '%smape%'
    AND `model` IS NOT NULL
    AND `metric_value` IS NOT NULL
  GROUP BY `model`
), ranked_models AS (
  SELECT
    `model`,
    `avg_smape`,
    RANK() OVER (ORDER BY `avg_smape` ASC) AS `smape_rank`
  FROM model_smape
)
SELECT
  `model`,
  `avg_smape`
FROM ranked_models
WHERE `smape_rank` <= 1
ORDER BY `avg_smape` ASC, `model` ASC

Answer:
The model with the lowest average sMAPE across all time series is **StatsForecastAutoCES**, with an average sMAPE o

In [0]:
# ── Step 7: App Serving Layer ────────────────────────────────────────
# This replicates exactly what apps/app.py does when a user selects a
# product and model in the Dash dashboard. The app connects to Lakebase
# PostgreSQL (not Spark) for sub-second response times.

import psycopg
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
ep = list(w.postgres.list_endpoints(
    parent="projects/many-model-forecasting-advanced/branches/production"
))[0]
cred = w.postgres.generate_database_credential(endpoint=ep.name)
host = ep.status.hosts.host
username = w.current_user.me().user_name

print("=== App Dashboard Query (served via Lakebase PostgreSQL) ===")
print(f"Endpoint: {host}")
print()

with psycopg.connect(host=host, dbname="databricks_postgres",
                      user=username, password=cred.token, sslmode="require") as conn:
    with conn.cursor() as cur:
        # Query 1: Model leaderboard (what the app's sidebar shows)
        cur.execute("""
            SELECT model, ROUND(AVG(metric_value)::numeric, 6) as avg_smape,
                   COUNT(DISTINCT unique_id) as series_count
            FROM m4_advanced."lb_monthly_evaluation_output"
            WHERE metric_name = 'smape'
            GROUP BY model ORDER BY avg_smape LIMIT 5
        """)
        print("Top 5 Models (app leaderboard view):")
        print(f"  {'Model':<42} {'Avg sMAPE':>10} {'Series':>7}")
        print(f"  {'-'*42} {'-'*10} {'-'*7}")
        for r in cur.fetchall():
            print(f"  {r[0]:<42} {r[1]:>10} {r[2]:>7}")

        # Query 2: Forecast for a specific product (what the chart shows)
        # y column is an array of forecast values; ds is array of dates
        cur.execute("""
            SELECT unique_id, model, y[1] as next_period_forecast
            FROM m4_advanced."lb_monthly_scoring_output"
            WHERE unique_id = 'M1'
            ORDER BY model LIMIT 5
        """)
        print(f"\nForecast for series M1 (app chart data, next-period value):")
        for r in cur.fetchall():
            print(f"  {r[0]} | {r[1]:<40} | forecast={r[2]:.2f}")

        # Query 3: Training data for sparkline (app time series view)
        cur.execute("""
            SELECT unique_id, MIN(ds) as start, MAX(ds) as end,
                   COUNT(*) as points, ROUND(AVG(y)::numeric, 1) as avg_demand
            FROM m4_advanced."lb_m4_monthly_train"
            GROUP BY unique_id
            ORDER BY unique_id LIMIT 5
        """)
        print(f"\nTraining data summary (app sparkline source):")
        for r in cur.fetchall():
            print(f"  {r[0]}: {r[3]} points, avg demand={r[4]}, range {r[1]} to {r[2]}")

print(f"\n✓ All 3 app queries executed against Lakebase PostgreSQL")
print(f"  These are the exact queries apps/app.py runs when serving the dashboard.")

=== App Dashboard Query (served via Lakebase PostgreSQL) ===
Endpoint: ep-noisy-rain-d1p0244n.database.us-west-2.cloud.databricks.com

Top 5 Models (app leaderboard view):
  Model                                       Avg sMAPE  Series
  ------------------------------------------ ---------- -------
  StatsForecastAutoCES                         0.061535      99
  StatsForecastAutoETS                         0.062436      99
  StatsForecastAutoArima                       0.063498      99
  SKTimeProphet                                0.066887      99
  StatsForecastAutoTheta                       0.068614      99

Forecast for series M1 (app chart data, next-period value):
  M1 | SKTimeProphet                            | forecast=4660.00
  M1 | StatsForecastADIDA                       | forecast=5096.53
  M1 | StatsForecastAutoArima                   | forecast=4654.77
  M1 | StatsForecastAutoCES                     | forecast=4533.70
  M1 | StatsForecastAutoETS                     | f

In [0]:
# ── Generate pipeline_run_log.md ──────────────────────────────────────
# This captures live outputs from every pipeline layer into a plain .md
# file that Git won't strip (unlike .ipynb cell outputs).

import datetime, psycopg, time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
now = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")
log = []
log.append(f"# Pipeline Run Log — {now}")
log.append("")
log.append("Automated evidence capture from `examples/evidence_runner` notebook.")
log.append("Every output below is a live query result, not mocked.")
log.append("")

# ── 1. Ingestion ──────────────────────────────────────────────────
log.append("## 1. Data Ingestion")
df = spark.table("mmf_demo_aym.m4_advanced.basf_chemicals_train")
count = df.count()
prods = df.select("unique_id").distinct().count()
dr = df.selectExpr("MIN(ds)", "MAX(ds)").collect()[0]
log.append(f"- Table: `mmf_demo_aym.m4_advanced.basf_chemicals_train`")
log.append(f"- Rows: **{count}** | Products: **{prods}** | Range: {dr[0]} to {dr[1]}")
log.append(f"- Exogenous regressors: feedstock_price_index, plant_utilization_rate, regulatory_compliance")
log.append("")

# ── 2. Governance ─────────────────────────────────────────────────
log.append("## 2. Unity Catalog Governance")
tags = spark.sql("""
  SELECT table_name, tag_name, tag_value
  FROM mmf_demo_aym.information_schema.table_tags
  WHERE schema_name = 'm4_advanced' ORDER BY table_name, tag_name
""").collect()
log.append(f"- Tags applied: **{len(tags)}** across governed tables")
log.append("```")
for r in tags:
    log.append(f"  {r.table_name}.{r.tag_name} = {r.tag_value}")
log.append("```")
cols = spark.sql("""
  SELECT column_name, comment FROM mmf_demo_aym.information_schema.columns
  WHERE table_schema='m4_advanced' AND table_name='basf_chemicals_train'
  ORDER BY ordinal_position
""").collect()
log.append("- Column comments (basf_chemicals_train):")
log.append("```")
for r in cols:
    log.append(f"  {r.column_name}: {r.comment}")
log.append("```")
log.append("")

# ── 3. MLflow Training ────────────────────────────────────────────
log.append("## 3. MLflow Training & Evaluation")
eval_df = spark.sql("""
  SELECT model, COUNT(*) as evals, ROUND(AVG(metric_value),6) as avg_smape
  FROM mmf_demo_aym.m4_advanced.monthly_evaluation_output
  GROUP BY model ORDER BY avg_smape
""")
rows = eval_df.collect()
run_id = spark.sql("SELECT DISTINCT run_id FROM mmf_demo_aym.m4_advanced.monthly_evaluation_output").collect()[0][0]
total_evals = sum(r.evals for r in rows)
log.append(f"- MLflow Run ID: `{run_id}`")
log.append(f"- Total evaluations: **{total_evals:,}** (17 models × 99 series × 10 windows)")
log.append("- Model leaderboard:")
log.append("```")
log.append(f"  {'Model':<44} {'Evals':>6} {'Avg sMAPE':>10}")
for r in rows:
    smape_str = f"{r.avg_smape:.6f}" if r.avg_smape else "N/A"
    log.append(f"  {r.model:<44} {r.evals:>6} {smape_str:>10}")
log.append("```")
log.append("")

# ── 4. Scoring ─────────────────────────────────────────────────────
log.append("## 4. Production Scoring")
scoring = spark.sql("""
  SELECT model, COUNT(DISTINCT unique_id) as series
  FROM mmf_demo_aym.m4_advanced.monthly_scoring_output GROUP BY model ORDER BY model
""").collect()
total_scored = sum(r.series for r in scoring)
log.append(f"- Total forecasts: **{total_scored:,}** across **{len(scoring)}** models")
log.append("")

# ── 5. Lakebase Serving ───────────────────────────────────────────
log.append("## 5. Lakebase Serving Layer")
ep = list(w.postgres.list_endpoints(
    parent="projects/many-model-forecasting-advanced/branches/production"))[0]
host = ep.status.hosts.host
cred = w.postgres.generate_database_credential(endpoint=ep.name)
username = w.current_user.me().user_name
log.append(f"- Project: many-model-forecasting-advanced")
log.append(f"- Host: `{host}`")
with psycopg.connect(host=host, dbname="databricks_postgres",
                      user=username, password=cred.token, sslmode="require") as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        ver = cur.fetchone()[0].split(",")[0]
        log.append(f"- Connected: {ver}")
        cur.execute("SELECT tablename FROM pg_tables WHERE schemaname='m4_advanced' ORDER BY 1")
        log.append("- Synced tables:")
        for r in cur.fetchall():
            cur.execute(f'SELECT COUNT(*) FROM m4_advanced.\"{r[0]}\"')
            log.append(f"  - `{r[0]}`: {cur.fetchone()[0]:,} rows")
        # App-style query
        cur.execute("""
            SELECT model, ROUND(AVG(metric_value)::numeric,6) as avg_smape
            FROM m4_advanced."lb_monthly_evaluation_output"
            WHERE metric_name='smape' GROUP BY model ORDER BY avg_smape LIMIT 3
        """)
        log.append("- App leaderboard query (top 3):")
        log.append("```")
        for r in cur.fetchall():
            log.append(f"  {r[0]}: sMAPE={r[1]}")
        log.append("```")
for tbl in ["lb_monthly_evaluation_output", "lb_monthly_scoring_output", "lb_m4_monthly_train"]:
    st = w.postgres.get_synced_table(name=f"synced_tables/mmf_demo_aym.m4_advanced.{tbl}")
    log.append(f"- Sync status `{tbl}`: **{st.status.detailed_state}**")
log.append("")

# ── 6. Genie Agent ────────────────────────────────────────────────
log.append("## 6. Genie Agent")
GENIE_SPACE_ID = "01f1c22c3f621207842ec9d6577c22c7"
log.append(f"- Space: MMF Demand Forecasting Intelligence")
log.append(f"- Space ID: `{GENIE_SPACE_ID}`")
log.append(f"- Question: *Which model has the lowest average sMAPE?*")
conv = w.genie.start_conversation(space_id=GENIE_SPACE_ID,
    content="Which model has the lowest average sMAPE across all time series?")
for _ in range(60):
    msg = w.genie.get_message(space_id=GENIE_SPACE_ID,
        conversation_id=conv.conversation_id, message_id=conv.message_id)
    if msg.status in ("COMPLETED", "FAILED"): break
    time.sleep(2)
log.append(f"- Status: **{msg.status}**")
if hasattr(msg, 'attachments') and msg.attachments:
    for att in msg.attachments:
        if hasattr(att, 'query') and att.query:
            log.append("- Generated SQL:")
            log.append("```sql")
            log.append(att.query.query)
            log.append("```")
        if hasattr(att, 'text') and att.text:
            log.append(f"- Answer: {att.text.content}")
log.append("")

# ── Write the log ─────────────────────────────────────────────────
log.append("---")
log.append(f"*Generated by `examples/evidence_runner` at {now}*")

import pathlib
out_path = pathlib.Path("/Workspace/Users/aliyener.mutlu@databricks.com/multi-model-demand-forecasting-git/examples/pipeline_run_log.md")
out_path.write_text("\n".join(log))
print(f"✓ Pipeline run log written to {out_path}")
print(f"  {len(log)} lines covering all 6 pipeline layers")
print(f"  This is a plain .md file — Git will preserve it with full outputs.")

✓ Pipeline run log written to /Workspace/Users/aliyener.mutlu@databricks.com/multi-model-demand-forecasting-git/examples/pipeline_run_log.md
  95 lines covering all 6 pipeline layers
  This is a plain .md file — Git will preserve it with full outputs.


## Summary — End-to-End Journey
> All outputs above are live-executed. Run log also written to `examples/pipeline_run_log.md`.

| Step | Layer | Evidence |
|------|-------|--------|
| 1 | **Ingestion & Transformation** | Raw data generated, Delta table written to UC |
| 2 | **UC Governance** | Table/column comments, tags verified via information_schema |
| 3 | **Data Verification** | 720 rows, 12 products, 3 exogenous regressors confirmed |
| 4 | **MLflow Training** | 17 models × 99 series × 10 windows = 16,830 evaluations |
| 5 | **Production Scoring** | 1,683 forecasts across 17 models |
| 6 | **Lakebase Serving** | 3 synced tables in PostgreSQL 17, all ONLINE |
| 7 | **Genie Agent** | Live conversation: SQL generated, answer returned |
| 8 | **App Serving** | 3 dashboard queries executed against Lakebase PostgreSQL |